In [593]:
from rdflib import Graph, URIRef
from rdflib.namespace import RDF, RDFS, OWL, SKOS
from dash import Dash, html, dcc, Input, Output, State, ctx
import dash_cytoscape as cyto
import json
from rdflib import Literal, Namespace
import random
import os
import re
import dash_draggable
import base64
import dash
from dash.exceptions import PreventUpdate

In [594]:
import requests
from pathlib import Path

url = "https://www.w3.org/2006/time.rdf"

output = Path("ontology/ontologies/time.rdf")
output.parent.mkdir(parents=True, exist_ok=True)

response = requests.get(url)
response.raise_for_status()

print("Content-Type:", response.headers.get("Content-Type"))
print("Size:", len(response.content))

output.write_bytes(response.content)

print(f"Saved to: {output}")

Content-Type: application/rdf+xml; qs=0.9
Size: 135294
Saved to: ontology/ontologies/time.rdf


In [595]:
g = Graph()

g.parse("ontology/HolyWells_Ontology.rdf")
g.parse("ontology/ontologies/time.rdf")

print(f"{len(g)} triples loaded")


SCHEMA = Namespace("http://schema.org/")
TIME = Namespace("http://www.w3.org/2006/time#")
OWL = Namespace("http://www.w3.org/2002/07/owl#")

2093 triples loaded


In [596]:
# get label
def label(resource):
    """
    Return a human-readable label for a resource or literal.

    Priority:
      1. English rdfs:label
      2. English skos:altLabel
      3. Any rdfs:label
      4. Any skos:altLabel
      5. URI fragment / last path segment
      6. Literal value
    """

    # If a plain string was passed, convert URI-like strings to URIRef
    if isinstance(resource, str):
        if resource.startswith("http://") or resource.startswith("https://"):
            resource = URIRef(resource)
        else:
            return  resource
    # Literal
    if isinstance(resource, Literal):
        return str(resource)
    
    # -----------------------------------------------------
    # rdfs:label — prefer English
    # -----------------------------------------------------

    labels = list(g.objects(resource, RDFS.label))

    english_labels = [
        lit for lit in labels
        if isinstance(lit, Literal)
        and lit.language
        and lit.language.lower().startswith("en")
    ]

    if english_labels:
        return str(english_labels[0])
    
    # Fallback to any label
    if labels:
        return str(labels[0])


   

    # rdfs:label
    lbl = g.value(resource, RDFS.label)
    if lbl is not None:
        return str(lbl)

    # skos:altLabel
    alt = g.value(resource, SKOS.altLabel)
    if alt is not None:
        return  str(alt)

    # Fallback to URI fragment
    text = str(resource)
#here
    if "#" in text:
        return text.rsplit("#", 1)[-1]

    return text.rstrip("/").rsplit("/", 1)[-1]

In [597]:
g = Graph()

g.parse("ontology/HolyWells_Ontology.rdf")
g.parse("ontology/ontologies/time.rdf")

print(f"{len(g)} triples loaded")


SCHEMA = Namespace("http://schema.org/")
TIME = Namespace("http://www.w3.org/2006/time#")
OWL = Namespace("http://www.w3.org/2002/07/owl#")


# =========================================================
# Get Classes
# =========================================================

classes = set(g.subjects(RDF.type, OWL.Class))

# RDFS classes
classes.update(
    g.subjects(RDF.type, RDFS.Class)
)

# Resources used as rdf:type of another resource
classes.update(
    obj
    for obj in g.objects(None, RDF.type)
    if isinstance(obj, URIRef)
)

print(f"Found {len(classes)} classes")


# =========================================================
# Datatypes
# =========================================================

datatypes = set(
    g.subjects(RDF.type, RDFS.Datatype)
)

datatypes_string = {
    str(d)
    for d in datatypes
}

print(f"Found {len(datatypes)} datatypes")


# =========================================================
# Named individuals
# =========================================================

named_individuals = set(
    g.subjects(RDF.type, OWL.NamedIndividual)
)

# =========================================================
# All resources explicitly typed with something
# =========================================================

typed_resources = set(
    g.subjects(RDF.type, None)
)


# =========================================================
# Individuals
#
# Anything explicitly typed that isn't a Class or Datatype
# is treated as an individual/resource.
# =========================================================

individuals = (
    typed_resources
    - classes
    - datatypes
)

print(f"Found {len(individuals)} individuals")


# =========================================================
# Unnamed individuals
# =========================================================

unnamed_individuals = (
    individuals - named_individuals
)

print(f"Named individuals: {len(named_individuals)}")
print(f"Unnamed individuals: {len(unnamed_individuals)}")


2093 triples loaded
Found 144 classes
Found 3 datatypes
Found 296 individuals
Named individuals: 96
Unnamed individuals: 200


In [598]:
# Convert resources to string sets for easy lookup
classes_string = {(str(c) )for c in classes}

individuals_string = {str(i) for i in individuals}

label_literals = {
    str(lit)
    for _, lit in g.subject_objects(RDFS.label)
    if isinstance(lit, Literal)
}

alt_label_literals = {
    str(lit)
    for _, lit in g.subject_objects(SKOS.altLabel)
    if isinstance(lit, Literal)
}

description_literals = {
    str(lit)
    for _, lit in g.subject_objects(SCHEMA.description)
    if isinstance(lit, Literal)
}

all_literals = {
    str(obj)
    for _, _, obj in g
    if isinstance(obj, Literal)
}

other_literals = (
    all_literals
    - label_literals
    - alt_label_literals
    - description_literals
)

In [599]:
def get_node_kind(value):
    """
    Classify a node value (URI string or literal string).
    """

    # Choose symbols
    CLASS_SYMBOL = "● "         #black circle
    INDIVIDUAL_SYMBOL = "◆ "    #black diamond
    LITERAL_SYMBOL = "▲ "       #black triangle
    LABEL_SYMBOL =    "■ "      #black square
    ALT_LABEL_SYMBOL= "★ "      #black hexagon
    DESCRIPTION_SYMBOL= "✚ "    #black pentagon
    DATATYPE_SYMBOL = "◇ "      # white diamond
    
    # Try to interpret the value as a URI resource
    try:
        uri = str(URIRef(value))
    except Exception:
        uri = None
    # ----- Literal categories -----

    if value in label_literals:
        return ["label", LABEL_SYMBOL]

    if value in alt_label_literals:
        return ["alt_label", ALT_LABEL_SYMBOL]

    if value in description_literals:
        return ["description", DESCRIPTION_SYMBOL]

    if value in other_literals:
        return ["literal", LITERAL_SYMBOL]

    if value in datatypes_string:
        return ["datatype", DATATYPE_SYMBOL]

    # ----- Resource categories -----

    if value in classes_string:
        return ["class", CLASS_SYMBOL]

    if value in individuals_string:
        return ["individual", INDIVIDUAL_SYMBOL]

    return ["resource", ""]

In [600]:
def get_predicates(resource):

    resource_ref = URIRef(resource)

    outgoing = sorted({
        str(predicate)
        for predicate in g.predicates(resource_ref, None)
    })

    incoming = sorted({
        str(predicate)
        for predicate in g.predicates(None, resource_ref)
    })

    print("\n" + "=" * 70)
    print("GET PREDICATES")
    print("=" * 70)
    print("resource:", resource_ref)

    print("\nOUTGOING:")
    for p in outgoing:
        print("  →", p)

    print("\nINCOMING:")
    for p in incoming:
        print("  ←", p)

    print("\nTOTAL:")
    print(" outgoing:", len(outgoing))
    print(" incoming:", len(incoming))

    print("=" * 70)

    return {
        "outgoing": outgoing,
        "incoming": incoming,
    }

In [601]:
get_predicates("http://www.wikidata.org/entity/June9_Pattern")


GET PREDICATES
resource: http://www.wikidata.org/entity/June9_Pattern

OUTGOING:
  → http://www.w3.org/1999/02/22-rdf-syntax-ns#type
  → http://www.w3.org/2000/01/rdf-schema#label
  → http://www.w3.org/2006/time#day
  → http://www.w3.org/2006/time#month

INCOMING:
  ← http://www.w3.org/2006/time#hasDateTimeDescription

TOTAL:
 outgoing: 4
 incoming: 1


{'outgoing': ['http://www.w3.org/1999/02/22-rdf-syntax-ns#type',
  'http://www.w3.org/2000/01/rdf-schema#label',
  'http://www.w3.org/2006/time#day',
  'http://www.w3.org/2006/time#month'],
 'incoming': ['http://www.w3.org/2006/time#hasDateTimeDescription']}

What I still want:

1. expand all option for each node
2. delete orphan nodes by default

5. export image as svg and png


In [602]:
# Dropdown options and nodes types
all_nodes_options =[]
classes_options = []
individuals_options = []
named_individuals_options =[]
unnamed_individuals_options =[]

# set up the options for the drop down menus, keep all options. 
resources = set()

for s, p, o in g:
    if isinstance(s, URIRef):
        resources.add(s)

    if isinstance(o, URIRef):
        resources.add(o)

for resource in resources:

    # Only URI resources - no blanks
    if not isinstance(resource, URIRef):
        continue

    all_nodes_options.append(
        {
            "label": f"{label(resource)}",
            "value": str(resource)
        }
    )

    if (resource in classes):
        classes_options.append(
            {
                "label": f"{label(resource)}",
                "value": str(resource)
            }
        )
    if (resource in named_individuals):
        named_individuals_options.append(
            {
                "label": f"{label(resource)}",
                "value": str(resource)
            }
        )

    if (resource in unnamed_individuals):
        unnamed_individuals_options.append(
            {
                "label": f"{label(resource)}",
                "value": str(resource)
            }
        )
#sort options alphabetically
classes_options.sort(key=lambda d:d["label"])
named_individuals_options.sort(key=lambda d:d["label"])
unnamed_individuals_options.sort(key=lambda d:d["label"])
all_nodes_options.sort(key= lambda d:d["label"])

type_options =["Class", "Named Individual", "Unnamed Individual"]


all_options = {'Class':classes_options, 'Named Individual':named_individuals_options, 'Unnamed Individual':unnamed_individuals_options}

In [603]:
# Helpers
# ============================================================

def hidden_delete_style():
    return {
        "display": "none",
        "position": "absolute",
        "zIndex": 1001,
    }


def hidden_menu_style():
    return {
        "display": "none",
    }


def node_controls_style(x, y):
    return {
        "display": "block",
        "position": "absolute",
        "left": f"{x + 15}px",
        "top": f"{y + 15}px",
        "zIndex": 1000,
        "pointerEvents": "auto",
    }


def delete_button_style(x, y):
    return {
        "display": "block",
        "position": "absolute",
        "left": f"{x + 15}px",
        "top": f"{y - 40}px",
        "zIndex": 1001,
        "pointerEvents": "auto",
    }

In [604]:
# Dash application style

app = Dash(__name__)

# ----------------------------------------------------
# Floating legend (put this BEFORE app.layout)
# ----------------------------------------------------
# ----------------------------------------------------
# Floating legend
# ----------------------------------------------------

def legend_row(symbol, text, color):
    return html.Div(
        [
            html.Span(
                symbol,
                className="legend-symbol",
            ),

            html.Span(
                text,
                className="legend-text",
            ),
        ],
        className="legend-row",
        style={
            "backgroundColor": color,
            "borderRadius": "8px",

            "padding": "6px 10px",
            "marginBottom": "6px",

            "color": "black",

            "border": "1px solid rgba(0,0,0,0.15)",

            "boxSizing": "border-box",
            "userSelect": "none",
        },
    )

legend = html.Div(
    [

        # ------------------------------------------------
        # Legend title
        # ------------------------------------------------

        html.Div(
            "Legend",
            id="legend-title",
            style={
                "fontWeight": "bold",
                "marginBottom": "10px",
                "padding": "8px",
                "backgroundColor": "#f0f0f0",
                "borderRadius": "6px",
                "textAlign": "center",
                "boxSizing": "border-box",
            },
        ),

        # ------------------------------------------------
        # Legend entries
        # ------------------------------------------------

        legend_row("●", "Class", "#FAC369"),
        legend_row("◆", "Individual", "#BCA7C7"),
        legend_row("■", "Label", "#B38D68"),
        legend_row("★", "Alternative label", "#ED9F77"),
        legend_row("✚", "Description", "#F9BC90"),
        legend_row("▲", "Literal", "#95DEA0"),
        legend_row("◇", "Datatype", "#DE9595"),

        # ------------------------------------------------
        # Resize handle
        # ------------------------------------------------

        html.Div(
            "↘",
            id="legend-resize-handle",
            title="Resize legend — double-click to return to automatic size",
            style={
                "position": "absolute",
                "right": "-7px",
                "bottom": "-7px",

                "width": "18px",
                "height": "18px",

                "backgroundColor": "white",
                "border": "2px solid #555",
                "borderRadius": "50%",

                "fontSize": "13px",
                "fontWeight": "bold",
                "lineHeight": "14px",
                "textAlign": "center",

                "cursor": "nwse-resize",

                "zIndex": 3001,

                "pointerEvents": "auto",

                "boxSizing": "border-box",
            },
        ),
    ],

    id="legend-box",

    style={
        "position": "absolute",

        "left": "20px",
        "top": "20px",

        "zIndex": 3000,

        "backgroundColor": "white",

        "border": "1px solid #888",

        "borderRadius": "10px",

        "padding": "10px",

        "boxShadow":
            "2px 2px 10px rgba(0,0,0,0.25)",

        # No fixed width.
        # JS controls the scale.
        "width": "max-content",

        "minWidth": "150px",

        "cursor": "move",

        "userSelect": "none",

        "boxSizing": "border-box",

        "transformOrigin": "top left",
    },
)

app.layout = html.Div(

    [

        html.H2("RDF Graph Browser"),
    
        # ----------------------------------------------------
        # Node Type Selector
        # ----------------------------------------------------

        dcc.Dropdown(

            id="type-dropdown",
            
            options=type_options,

            placeholder="Choose Type",

            searchable=True,

            clearable=True,

            style={
                "width":"60%"
            }

        ),

        html.Br(),

        # ----------------------------------------------------
        # Node Selector
        # ----------------------------------------------------

        dcc.Dropdown(

            id="node-dropdown",
            
            options=all_nodes_options,

            placeholder="Choose starting node",

            searchable=True,

            clearable=True,

            style={
                "width":"60%"
            }

        ),

        html.Br(),

        # ----------------------------------------------------
        # Save and upload button
        # ----------------------------------------------------
        dcc.Input(
            id="save-filename",
            type="text",
            placeholder="graph_name.json",
            value="my_graph",
            style={"width": "250px", "marginRight": "10px"}
        ),

        html.Button( "Save graph layout", id="save-as-button"),

        html.Button("Load graph layout", id="load-graph", style={"marginLeft": "10px"}),

         html.Button("Export PNG", id="export-png-button", style={"marginLeft": "10px"}
        ),

        html.Div(id="save-status", style={"marginTop": "10px"}),

       
        html.Br(),

        # ----------------------------------------------------
        # Edge Expansion Selector, draggable legend and graph
        # under same div so selector is placed relative to node
        # ----------------------------------------------------
        html.Div(
            [
        

            
                # ----------------------------------------------------
                # Cytoscape graph style
                # ----------------------------------------------------

                cyto.Cytoscape(

                    id="graph",
                    elements=[],

                    #initial zoom
                    zoom=0.9,

                    # Initial pan
                    pan={"x": 0, "y": 0},

                    layout={
                        "name": "preset",
                        "fit": False,
                        "animate": False
                    },
                    style={
                        "width":"100%",
                        "height":"900px"
                    },

                    stylesheet=[
                        # --------------------------------------------
                        # Nodes
                        # --------------------------------------------
                        {
                            "selector":"node[kind='class']",

                            "style":
                            {
                                "label": "data(label)",
                                "background-color": "#FAC369", #yellow
                                "color": "black",
                                "shape": "round-rectangle",
                            
                                "width": "label",
                                "height": "label",

                                "padding": "10px",
                                "padding-left": "24px",
                                "padding-right": "10px",
                                "padding-top": "10px",
                                "padding-bottom": "10px",

                                "text-wrap": "wrap",
                                "text-max-width": "120px",

                                "text-valign": "center",
                                "text-halign": "center",

                                "font-size": 15,

                            }

                        },

                        {
                            "selector":"node[kind='individual']",

                            "style":
                            {
                                "label": "data(label)",
                                "background-color": "#BCA7C7", #"#6D3D80", # purple
                                "color": "black",

                                "shape": "round-rectangle",

                                "width": "label",
                                "height": "label",

                                "padding": "10px",
                                "padding-left": "24px",
                                "padding-right": "10px",
                                "padding-top": "10px",
                                "padding-bottom": "10px",

                                "text-wrap": "wrap",
                                "text-max-width": "120px",

                                "text-valign": "center",
                                "text-halign": "center",

                                "font-size": 15,
                            }

                        },

                        {
                            "selector":"node[kind='label']",

                            "style":
                            {
                                "label": "data(label)",
                                "background-color": "#B38D68", # orange
                                "color": "black",

                                "shape": "round-rectangle",

                                "width": "label",
                                "height": "label",

                                "padding": "10px",
                                "padding-left": "24px",
                                "padding-right": "10px",
                                "padding-top": "10px",
                                "padding-bottom": "10px",

                                "text-wrap": "wrap",
                                "text-max-width": "120px",

                                "text-valign": "center",
                                "text-halign": "center",

                                "font-size": 15

                            }

                        },
                        {
                            "selector":"node[kind='alt_label']",

                            "style":
                            {
                                "label": "data(label)",
                                "background-color": "#ED9F77", # tangerine
                                "color": "black",

                                "shape": "round-rectangle",

                                "width": "label",
                                "height": "label",

                                "padding": "10px",
                                "padding-left": "24px",
                                "padding-right": "10px",
                                "padding-top": "10px",
                                "padding-bottom": "10px",

                                "text-wrap": "wrap",
                                "text-max-width": "120px",

                                "text-valign": "center",
                                "text-halign": "center",

                                "font-size": 15

                            }

                        },


                        {
                            "selector":"node[kind='description']",

                            "style":
                            {
                                "label": "data(label)",
                                "background-color": "#F9BC90", # peach
                                "color": "black",

                                "shape": "round-rectangle",

                                "width": "label",
                                "height": "label",
                                
                                "padding": "10px",
                                "padding-left": "24px",
                                "padding-right": "10px",
                                "padding-top": "10px",
                                "padding-bottom": "10px",

                                "text-wrap": "wrap",
                                "text-max-width": "120px",

                                "text-valign": "center",
                                "text-halign": "center",

                                "font-size": 15

                            }

                        },

                        {
                            "selector":"node[kind='literal']",

                            "style":
                            {
                                "label": "data(label)",
                                "background-color": "#95DEA0", # green 
                                "color": "black",

                                "shape": "round-rectangle",

                                "width": "label",
                                "height": "label",

                                "padding": "10px",
                                "padding-left": "24px",
                                "padding-right": "10px",
                                "padding-top": "10px",
                                "padding-bottom": "10px",

                                "text-wrap": "wrap",
                                "text-max-width": "120px",

                                "text-valign": "center",
                                "text-halign": "center",

                                "font-size": 15

                            }

                        },

                        {
                            "selector":"node[kind='datatype']",

                            "style":
                            {
                                "label": "data(label)",
                                "background-color": "#DE9595", # red
                                "color": "black",

                                "shape": "round-rectangle",

                                "width": "label",
                                "height": "label",

                                "padding": "10px",
                                "padding-left": "24px",
                                "padding-right": "10px",
                                "padding-top": "10px",
                                "padding-bottom": "10px",

                                "text-wrap": "wrap",
                                "text-max-width": "120px",

                                "text-valign": "center",
                                "text-halign": "center",

                                "font-size": 15

                            }

                        },

                        {
                            "selector": "edge[kind='rdf_edge']",
                            "style": {
                                "label": "data(label)",
                                "curve-style": "bezier",
                                "target-arrow-shape": "triangle",
                                "width": 2,
                                "font-size": 12,
                                "text-background-color": "white",
                                "text-background-opacity": 1
                            }
                        },
                    ]

                ),
                
                
                # ----------------------------------------------------
                # 12 x 12 cm PNG export frame
                # ----------------------------------------------------
                html.Div(
                    id="png-export-frame",
                    children=[

                        # The visible frame itself does NOT receive clicks.
                        html.Div(
                            id="png-export-frame-label",
                        ),

                        # ------------------------------------------------
                        # Dedicated drag handle
                        # ------------------------------------------------
                        html.Div(
                            "↔",
                            id="png-export-frame-handle",
                            title="Drag export area",
                            style={
                                "position": "absolute",
                                "left": "50%",
                                "top": "-14px",
                                "transform": "translateX(-50%)",

                                "width": "32px",
                                "height": "24px",

                                "backgroundColor": "white",
                                "border": "2px solid #ff0000",
                                "borderRadius": "6px",

                                "color": "#ff0000",
                                "fontWeight": "bold",
                                "fontSize": "18px",
                                "lineHeight": "20px",
                                "textAlign": "center",

                                "cursor": "move",

                                # THIS element receives the drag interaction
                                "pointerEvents": "auto",

                                "zIndex": 3001,

                                "boxSizing": "border-box",
                            },
                        ),
                    ],

                    style={
                        "position": "absolute",
                        "left": "20px",
                        "top": "20px",

                        "boxSizing": "border-box",

                        "border": "2px dashed #ff0000",

                        # Very subtle background
                        "backgroundColor": "rgba(255, 0, 0, 0.03)",

                        # The frame is behind Cytoscape
                        "zIndex": 1,

                        # CRITICAL:
                        # The frame itself does not intercept clicks.
                        "pointerEvents": "none",

                        # Size is set by draggable_legend.js
                    },
                ),
            
                # =========================================================
                # Floating controls
                # =========================================================
                html.Div(
                    [
                        html.Button(
                            "Delete selected",
                            id="delete-button",
                            disabled=True,
                            style=hidden_delete_style(),
                            
                        ),

                        dcc.Dropdown(
                            id="predicate-menu",
                            options=[],
                            value=None,
                            placeholder="Expand on...",
                            style=hidden_menu_style(),
                        ),
                    ],

                    id="floating-controls",
                    style={
                        "position": "absolute",
                        "top": "0px",
                        "left": "0px",
                        "width": "100%",
                        "height": "100%",
                        "zIndex": 1000,
                        # The overlay itself does not intercept Cytoscape clicks.
                        "pointerEvents": "none",
                    },
                ),

                dcc.Store(id="clicked-resource"),
                dcc.Store(id="selected-element"),
                dcc.Store(id="debug-store"),
                dcc.Store(id="png-export-data"),


                legend,

            ],
            id="graph-export-container",
            style={
                "position": "relative",
                "width": "100%",
                "height": "900px",
            },
        )

    ]


)

In [605]:
# Callback handle selection
@app.callback(
    Output("selected-element", "data"),

    Output("delete-button", "disabled"),
    Output("delete-button", "style"),

    Output("predicate-menu", "options"),
    Output("predicate-menu", "style"),
    Output("predicate-menu", "value"),

    Output("clicked-resource", "data"),

    Output("graph", "elements"), #allow_duplicate=True

    Output("debug-store", "data"),

    Input("graph", "tapNode"),
    Input("graph", "tapEdge"),
    Input("delete-button", "n_clicks"),
    Input("predicate-menu", "value"),


    State("selected-element", "data"),
    State("clicked-resource", "data"),
    State("graph", "elements"),

    #prevent_initial_call=True,
)
def handle_selection(
    node_data,
    edge_data,
    n_clicks,
    predicate_selection,
    selected,
    resource,
    elements,
):
    triggered = ctx.triggered_prop_ids

    debug_data = {
        "triggered": list(triggered.keys()),
        "node_data": node_data,
        "edge_data": edge_data,
        "selected_before": selected,
        "resource": resource,
        "predicate_selection": predicate_selection,
    }

    # ========================================================
    # DELETE
    # ========================================================

    if "delete-button.n_clicks" in triggered:

        if selected:

            selected_id = selected["id"]

            # --------------------------------------------
            # Delete selected node
            # --------------------------------------------

            if selected["type"] == "node":

                elements = [
                    e
                    for e in elements
                    if e["data"].get("id") != selected_id
                    and e["data"].get("source") != selected_id
                    and e["data"].get("target") != selected_id
                ]

            # --------------------------------------------
            # Delete selected edge
            # --------------------------------------------

            elif selected["type"] == "edge":

                elements = [
                    e
                    for e in elements
                    if e["data"].get("id") != selected_id
                ]

        # --------------------------------------------
        # Delete closes UI
        # --------------------------------------------

        return (
            None,                  # selected-element
            True,                  # delete-button.disabled
            hidden_delete_style(), # delete-button.style

            [],                    # predicate-menu.options
            hidden_menu_style(),   # predicate-menu.style
            None,                  # predicate-menu.value

            None,                  # clicked-resource

            elements,              # graph.elements

            debug_data,            # debug-store
        )

    # ========================================================
    # PREDICATE EXPANSION
    # ========================================================

    if "predicate-menu.value" in triggered:

        if predicate_selection and resource:

            direction, predicate = predicate_selection.split("::", 1)

            elements = add_single_predicate(
                resource=resource,
                predicate=predicate,
                direction=direction,
                elements=elements,
            )

        # --------------------------------------------
        # Expansion closes UI
        # --------------------------------------------

        return (
            None,                  # selected-element
            True,                  # delete-button.disabled
            hidden_delete_style(), # delete-button.style

            [],                    # predicate-menu.options
            hidden_menu_style(),   # predicate-menu.style
            None,                  # predicate-menu.value

            None,                  # clicked-resource

            elements,              # graph.elements

            debug_data,            # debug-store
        )

    # ========================================================
    # NODE CLICK
    # ========================================================

    if "graph.tapNode" in triggered and node_data:

        data = node_data["data"]
        pos = node_data["renderedPosition"]

        node_id = data["id"]

        # --------------------------------------------
        # Predicate options
        # --------------------------------------------

        predicates = get_predicates(node_id)

        options = []

        for predicate in predicates["outgoing"]:
            options.append(
                {
                    "label": f"→ {label(URIRef(predicate))}",
                    "value": f"out::{predicate}",
                }
            )

        for predicate in predicates["incoming"]:
            options.append(
                {
                    "label": f"← {label(URIRef(predicate))}",
                    "value": f"in::{predicate}",
                }
            )

        # --------------------------------------------
        # Selected node
        # --------------------------------------------

        selected_element = {
            "type": "node",
            "id": node_id,
        }

        return (
            selected_element,

            False,
            delete_button_style(
                pos["x"],
                pos["y"],
            ),

            options,
            node_controls_style(
                pos["x"],
                pos["y"],
            ),

            None,       # predicate-menu.value
            node_id,    # clicked-resource

            elements,

            debug_data,
        )

    # ========================================================
    # EDGE CLICK
    # ========================================================

    if "graph.tapEdge" in triggered and edge_data:

        edge_id = edge_data["data"]["id"]

        selected_element = {
            "type": "edge",
            "id": edge_id,
        }

        source_id = edge_data["data"]["source"]
        target_id = edge_data["data"]["target"]

        source_node = next(
            e for e in elements
            if e["data"].get("id") == source_id
        )

        target_node = next(
            e for e in elements
            if e["data"].get("id") == target_id
        )

        source_pos = source_node["position"]
        target_pos = target_node["position"]

        edge_x = (source_pos["x"] + target_pos["x"]) / 2
        edge_y = (source_pos["y"] + target_pos["y"]) / 2

        return (
            selected_element,

            False,
            delete_button_style(edge_x, edge_y),

            [],
            hidden_menu_style(),
            None,

            None,

            elements,

            debug_data,
        )

    # ========================================================
    # FALLBACK
    # ========================================================

    return (
        selected,

        selected is None,
        hidden_delete_style(),

        [],
        hidden_menu_style(),
        None,

        resource,

        elements,

        debug_data,
    )

In [606]:
# Callback Expand predicate

@app.callback(
    Output("graph", "elements", allow_duplicate=True),
    
    Input("predicate-menu", "value"),

    State("clicked-resource", "data"),
    State("graph", "elements"),

    prevent_initial_call=True,
)

def expand_selected_predicate(selection, resource, elements):

    if not selection or not resource:
        return (
            elements
        )

    direction, predicate = selection.split("::", 1)

    elements = add_single_predicate(
        resource=resource,
        predicate=predicate,
        direction=direction,
        elements=elements,
    )

    # ========================================================
    # Expansion is complete:
    # clear all selection/UI state
    # ========================================================
    
    return (
        elements
    )

def add_node(
    elements,
    node_id,
    label_text,
    kind,
    datatype=None,
):
    """
    Add a Cytoscape node.

    Parameters
    ----------
    elements : list
        Existing Cytoscape elements.

    node_id : str
        Unique Cytoscape node ID.

    label_text : str
        Text displayed in the graph.

    kind : str
        Node kind:
        class, individual, literal, resource, etc.

    datatype : str or None
        RDF datatype URI for a literal.
    """

    # Don't add the same node twice
    if any(
        e.get("data", {}).get("id") == node_id
        for e in elements
    ):
        return

    elements.append({
        "data": {
            "id": node_id,
            "label": label_text,
            "kind": kind,

            # Important for literals
            "datatype": datatype,
        },

        "position": {
            "x": random.randint(0, 200),
            "y": random.randint(0, 100),
        }
    })

def make_literal_id(literal):
    """
    Create a unique Cytoscape node ID for an RDF literal.

    The datatype is included because:
        "9"^^time:generalDay
    and
        "9"^^xsd:integer

    are different RDF literals.
    """

    value = str(literal)
    datatype = str(literal.datatype) if literal.datatype else ""
    language = literal.language or ""

    return f"literal::{value}::{datatype}::{language}"

def add_literal_datatype_node(
    elements,
    literal_node_id,
    literal,
    existing_nodes,
    existing_edges,
):
    """
    Add the RDF datatype node and a datatype edge
    for a typed literal.
    """

    if not literal.datatype:
        return

    datatype_ref = URIRef(str(literal.datatype))
    datatype_id = str(datatype_ref)

    # =====================================================
    # Add datatype node
    # =====================================================

    if datatype_id not in existing_nodes:

        datatype_kind, datatype_symbol = get_node_kind(
            datatype_id
        )

        add_node(
            elements=elements,
            node_id=datatype_id,
            label_text=(
                datatype_symbol
                + label(datatype_ref)
            ),
            kind=datatype_kind,
        )

        existing_nodes.add(datatype_id)

    # =====================================================
    # Add datatype edge
    # =====================================================

    edge = (
        literal_node_id,
        datatype_id,
        "datatype",
    )

    if edge not in existing_edges:

        elements.append({
            "data": {
                "id": (
                    f"{literal_node_id}"
                    f"|datatype|"
                    f"{datatype_id}"
                ),

                "source": literal_node_id,
                "target": datatype_id,

                "predicate": "datatype",

                "label": "datatype",

                "kind": "datatype_edge",
            }
        })

        existing_edges.add(edge)

# =========================================================
# Add single predicate
# =========================================================

def add_single_predicate(
    resource,
    predicate,
    direction,
    elements,
):

    # =====================================================
    # Convert resource and predicate to URIRefs
    #
    # IMPORTANT:
    # resource is expected to be a URI resource here.
    # Literals are handled as targets/sources below.
    # =====================================================

    resource_ref = URIRef(resource)

    predicate_ref = URIRef(predicate)


    print("\n" + "=" * 70)
    print("EXPANDING PREDICATE")
    print("=" * 70)

    print("RESOURCE :", resource_ref)
    print("PREDICATE:", predicate_ref)
    print("DIRECTION:", direction)


    # =====================================================
    # Existing nodes
    # =====================================================

    existing_nodes = {

        e["data"]["id"]

        for e in elements

        if "id" in e.get("data", {})
    }


    # =====================================================
    # Existing edges
    # =====================================================

    existing_edges = {

        (
            e["data"].get("source"),
            e["data"].get("target"),
            e["data"].get("predicate"),
        )

        for e in elements

        if "source" in e.get("data", {})
    }


    # =====================================================
    # IMPORTANT:
    #
    # Make sure the resource itself exists as a node.
    #
    # This is what fixes:
    #
    # "nonexistant source June9_Pattern"
    #
    # when expanding rdf:type.
    # =====================================================

    resource_id = str(resource_ref)


    if resource_id not in existing_nodes:

        resource_kind, resource_symbol = (
            get_node_kind(resource_id)
        )

        add_node(

            elements=elements,

            node_id=resource_id,

            label_text=(
                resource_symbol
                + label(resource_ref)
            ),

            kind=resource_kind,

            datatype=None,
        )

        existing_nodes.add(
            resource_id
        )


    # =====================================================
    # Get triples
    # =====================================================

    if direction == "out":

        triples = [

            (
                resource_ref,
                obj
            )

            for obj in g.objects(
                resource_ref,
                predicate_ref
            )
        ]


    elif direction == "in":

        triples = [

            (
                subj,
                resource_ref
            )

            for subj in g.subjects(
                predicate_ref,
                resource_ref
            )
        ]


    else:

        print(
            "ERROR: invalid direction:",
            direction
        )

        return elements


    print(
        "TRIPLES FOUND:",
        len(triples)
    )


    # =====================================================
    # Process every triple
    # =====================================================

    for source_ref, target_ref in triples:

        print("\nTRIPLE")

        print(
            "  source:",
            source_ref
        )

        print(
            "  predicate:",
            predicate_ref
        )

        print(
            "  target:",
            target_ref
        )


        # =================================================
        # SOURCE
        # =================================================

        if isinstance(
            source_ref,
            Literal
        ):

            # ---------------------------------------------
            # Source is an RDF literal
            # ---------------------------------------------

            source = make_literal_id(
                source_ref
            )

            source_label = str(
                source_ref
            )

            source_kind = "literal"

            source_symbol = "▲ "

            source_datatype = (

                str(source_ref.datatype)

                if source_ref.datatype

                else None
            )


        else:

            # ---------------------------------------------
            # Source is an RDF resource
            # ---------------------------------------------

            source = str(
                source_ref
            )

            source_kind, source_symbol = (
                get_node_kind(source)
            )

            source_label = label(
                source_ref
            )

            source_datatype = None


        # =================================================
        # TARGET
        # =================================================

        if isinstance(
            target_ref,
            Literal
        ):

            # ---------------------------------------------
            # IMPORTANT:
            #
            # Keep the literal as a Literal.
            #
            # DO NOT do:
            #
            # URIRef(target_ref)
            #
            # because 9 and 6 are RDF literals.
            # ---------------------------------------------

            target = make_literal_id(
                target_ref
            )

            target_label = str(
                target_ref
            )

            target_kind = "literal"

            target_symbol = "▲ "

            target_datatype = (

                str(target_ref.datatype)

                if target_ref.datatype

                else None
            )


        else:

            # ---------------------------------------------
            # Target is an RDF resource
            # ---------------------------------------------

            target = str(
                target_ref
            )

            target_kind, target_symbol = (
                get_node_kind(target)
            )

            target_label = label(
                target_ref
            )

            target_datatype = None


        # =================================================
        # ADD SOURCE NODE
        # =================================================

        if source not in existing_nodes:

            add_node(

                elements=elements,

                node_id=source,

                label_text=(
                    source_symbol
                    + source_label
                ),

                kind=source_kind,

                datatype=source_datatype,
            )

            existing_nodes.add(
                source
            )


            # ---------------------------------------------
            # If source is a typed literal, show datatype
            # ---------------------------------------------

            if isinstance(
                source_ref,
                Literal
            ):

                add_literal_datatype_node(

                    elements=elements,

                    literal_node_id=source,

                    literal=source_ref,

                    existing_nodes=existing_nodes,

                    existing_edges=existing_edges,
                )


        # =================================================
        # ADD TARGET NODE
        # =================================================

        if target not in existing_nodes:

            add_node(

                elements=elements,

                node_id=target,

                label_text=(
                    target_symbol
                    + target_label
                ),

                kind=target_kind,

                datatype=target_datatype,
            )

            existing_nodes.add(
                target
            )


            # ---------------------------------------------
            # If target is a typed literal, show datatype
            # ---------------------------------------------

            if isinstance(
                target_ref,
                Literal
            ):

                add_literal_datatype_node(

                    elements=elements,

                    literal_node_id=target,

                    literal=target_ref,

                    existing_nodes=existing_nodes,

                    existing_edges=existing_edges,
                )


        # =================================================
        # ADD RDF EDGE
        # =================================================

        edge = (

            source,

            target,

            str(predicate_ref),
        )


        if edge not in existing_edges:

            elements.append({

                "data": {

                    "id": (
                        f"{source}|"
                        f"{predicate_ref}|"
                        f"{target}"
                    ),

                    "source": source,

                    "target": target,

                    "predicate": str(
                        predicate_ref
                    ),

                    "label": label(
                        predicate_ref
                    ),

                    "kind": "rdf_edge",
                }
            })

            existing_edges.add(
                edge
            )


    print(
        "=" * 70
    )


    return elements

In [607]:
# Callback Dropdown Type Picker

@app.callback(

   Output(
        "node-dropdown",
        "options"
    ),

    Input(
        "type-dropdown",
        "value"
    )

)

def set_node_value_options(selected_type):
    if(selected_type is None):
        return all_nodes_options
    return all_options[selected_type]




In [608]:
# Callback Dropdown Start Node Picker

@app.callback(
    Output("graph", "elements", allow_duplicate=True),
    Output("graph", "zoom", allow_duplicate=True),
    Output("graph", "pan", allow_duplicate=True),

    Input("node-dropdown", "value"),

    prevent_initial_call=True,
)
def create_start_graph(uri):

    if uri is None:
        raise dash.exceptions.PreventUpdate

    elements = []

    add_node(
        elements,
        node_id=uri,
        label_text=get_node_kind(uri)[1] + label(URIRef(uri)),
        kind=get_node_kind(uri)[0]
    )

    return (
        elements,
        1,
        {"x": 0, "y": 0}
    )

In [609]:
# Callback Save Layout

@app.callback(
    Output("save-status", "children"),
    Input("save-as-button", "n_clicks"),
    State("save-filename", "value"),
    State("graph", "elements"),
    State("graph", "pan"),
    State("graph", "zoom"),
    prevent_initial_call=True,
)
def save_graph_as(n_clicks, filename, elements, pan, zoom):

    if not filename:
        return "Please enter a filename."

    # Remove illegal filename characters
    filename = re.sub(r'[\\\\/:*?\"<>|]', "_", filename.strip())

    # Add .json 
    if not filename.lower().endswith(".json"):
        filename += ".json"

    # Create folder if needed
    os.makedirs("saved_graphs", exist_ok=True)

    path = os.path.join("saved_graphs", filename)

    state = {
        "elements": elements,
        "pan": pan,
        "zoom": zoom,
    }

    with open(path, "w", encoding="utf-8") as f:
        json.dump(state, f, indent=2)

    return f"Saved graph to: {path}"


In [610]:
# Callback Save PNG
@app.callback(
    Output(
        "save-status",
        "children",
        allow_duplicate=True
    ),
    Input(
        "png-export-data",
        "data"
    ),
    State(
        "save-filename",
        "value"
    ),
    prevent_initial_call=True,
)
def save_png_file(image_data, filename):

    if not image_data:
        return "PNG export failed: no image data received."

    if not filename:
        return "Please enter a filename."

    # --------------------------------------------------------
    # Clean filename
    # --------------------------------------------------------

    filename = re.sub(
        r'[\\/:*?"<>|]',
        "_",
        filename.strip()
    )

    # Remove .json
    if filename.lower().endswith(".json"):
        filename = filename[:-5]

    # Remove .png
    if filename.lower().endswith(".png"):
        filename = filename[:-4]

    # Final filename
    png_filename = filename + ".png"


    # --------------------------------------------------------
    # Create output folder
    # --------------------------------------------------------

    os.makedirs(
        "saved_images",
        exist_ok=True
    )


    # --------------------------------------------------------
    # Full path
    # --------------------------------------------------------

    path = os.path.join(
        "saved_images",
        png_filename
    )


    # --------------------------------------------------------
    # Decode data URL
    # --------------------------------------------------------

    try:

        header, encoded = image_data.split(
            ",",
            1
        )

        image_bytes = base64.b64decode(
            encoded
        )

    except Exception as e:

        return f"PNG export failed: {e}"


    # --------------------------------------------------------
    # Write PNG
    # --------------------------------------------------------

    try:

        with open(
            path,
            "wb"
        ) as f:

            f.write(image_bytes)

    except Exception as e:

        return f"PNG export failed while saving: {e}"


    return f"Saved PNG to: {path}"

In [611]:
# Callback Load Saved Layout

@app.callback(
    Output("graph", "elements", allow_duplicate=True),
    Output("graph", "pan", allow_duplicate=True),
    Output("graph", "zoom", allow_duplicate=True),
    Input("load-graph", "n_clicks"),
    State("save-filename", "value"),
    prevent_initial_call=True,
)
def load_graph_named(n_clicks, filename):

    if not filename:
        raise dash.exceptions.PreventUpdate

    if not filename.lower().endswith(".json"):
        filename += ".json"

    path = os.path.join("saved_graphs", filename)

    if not os.path.exists(path):
        raise dash.exceptions.PreventUpdate

    with open(path, "r", encoding="utf-8") as f:
        state = json.load(f)

    return (
        state.get("elements", []),
        state.get("pan", {"x": 800, "y": -800}),
        state.get("zoom", 0.9),
    )

In [612]:
# RUN SERVER
# ============================================================


if __name__ == "__main__":

    app.run(
        debug=True
    )


GET PREDICATES
resource: http://ontology.holywells.link/ontology/E65_Q126528008

OUTGOING:
  → http://www.cidoc-crm.org/cidoc-crm/P14_carried_out_by
  → http://www.cidoc-crm.org/cidoc-crm/P4_has_time-span
  → http://www.cidoc-crm.org/cidoc-crm/P94_has_created
  → http://www.w3.org/1999/02/22-rdf-syntax-ns#type

INCOMING:

TOTAL:
 outgoing: 4
 incoming: 0

EXPANDING PREDICATE
RESOURCE : http://ontology.holywells.link/ontology/E65_Q126528008
PREDICATE: http://www.cidoc-crm.org/cidoc-crm/P14_carried_out_by
DIRECTION: out
TRIPLES FOUND: 1

TRIPLE
  source: http://ontology.holywells.link/ontology/E65_Q126528008
  predicate: http://www.cidoc-crm.org/cidoc-crm/P14_carried_out_by
  target: http://www.wikidata.org/entity/Q548721

GET PREDICATES
resource: http://www.wikidata.org/entity/Q548721

OUTGOING:
  → http://www.w3.org/1999/02/22-rdf-syntax-ns#type
  → http://www.w3.org/2000/01/rdf-schema#label

INCOMING:
  ← http://www.cidoc-crm.org/cidoc-crm/P14_carried_out_by

TOTAL:
 outgoing: 2
 inc

literal::British Ordnance Survey:::: does not look like a valid URI, trying to serialize this will break.



GET PREDICATES
resource: literal::British Ordnance Survey::::

OUTGOING:

INCOMING:

TOTAL:
 outgoing: 0
 incoming: 0


literal::British Ordnance Survey:::: does not look like a valid URI, trying to serialize this will break.



GET PREDICATES
resource: literal::British Ordnance Survey::::

OUTGOING:

INCOMING:

TOTAL:
 outgoing: 0
 incoming: 0

GET PREDICATES
resource: http://www.wikidata.org/entity/Q548721

OUTGOING:
  → http://www.w3.org/1999/02/22-rdf-syntax-ns#type
  → http://www.w3.org/2000/01/rdf-schema#label

INCOMING:
  ← http://www.cidoc-crm.org/cidoc-crm/P14_carried_out_by

TOTAL:
 outgoing: 2
 incoming: 1

GET PREDICATES
resource: http://www.wikidata.org/entity/Q548721

OUTGOING:
  → http://www.w3.org/1999/02/22-rdf-syntax-ns#type
  → http://www.w3.org/2000/01/rdf-schema#label

INCOMING:
  ← http://www.cidoc-crm.org/cidoc-crm/P14_carried_out_by

TOTAL:
 outgoing: 2
 incoming: 1

EXPANDING PREDICATE
RESOURCE : http://www.wikidata.org/entity/Q548721
PREDICATE: http://www.w3.org/1999/02/22-rdf-syntax-ns#type
DIRECTION: out
TRIPLES FOUND: 2

TRIPLE
  source: http://www.wikidata.org/entity/Q548721
  predicate: http://www.w3.org/1999/02/22-rdf-syntax-ns#type
  target: http://www.w3.org/2002/07/owl#Named

In [ ]:
http://127.0.0.1:8050/